Reading Resume.csv...
Extracting features from Resume_str...
Injecting anomalous profiles for fraud classification training...
Success! Created 'processed_resume_data.csv' with tailored columns.


In [4]:
import pandas as pd
import random
import re

# Set random seed for perfectly reproducible datasets
random.seed(42)

print("============ STARTING DATA PREPARATION ============")

# 1. Load the raw Resume.csv file
try:
    print("Reading raw 'Resume.csv' dataset...")
    df_raw = pd.read_csv("Resume.csv")
    print(f"Successfully loaded {len(df_raw)} records from raw data.")
except FileNotFoundError:
    print("Error: 'Resume.csv' not found in the current directory. Please place it here.")
    exit()

# 2. Isolate and clean text strings
df = pd.DataFrame()
df['resume_text'] = df_raw['Resume_str'].astype(str).str.strip()
df['category'] = df_raw['Category'].astype(str)

# 3. Feature Engineering: Extract Baseline Skill Counts
skills_lookup = ["python", "java", "react", "sql", "aws", "docker", "javascript", "html", "css", "c++", "linux", "excel", "tableau"]

def calculate_skills(text):
    text_lower = text.lower()
    return sum(1 for skill in skills_lookup if re.search(r'\b' + re.escape(skill) + r'\b', text_lower))

print("Extracting baseline skill counts from unstructured text...")
df['skills_count'] = df['resume_text'].apply(calculate_skills)

# Generate an organic baseline for experience
df['years_experience'] = df['resume_text'].apply(lambda x: min(round(len(x) / 600 + random.randint(-1, 2)), 15))
df['years_experience'] = df['years_experience'].clip(lower=1) 

# 4. Inject Complex Anomalies with Statistical Noise
fraud_phrases = [
    " World-class ultimate programming guru with cosmic execution capabilities. ",
    " Unparalleled master of all known coding languages and absolute visionary. ",
    " Globally recognized top-tier expert capable of absolute architectural magic. "
]

labels = []
processed_texts = []

print("Injecting complex anomalies with statistical noise...")
for idx, row in df.iterrows():
    dice_roll = random.random()
    
    if dice_roll < 0.20:
        # 1. OBVIOUS FRAUD (Easy to catch)
        modified_text = row['resume_text'] + random.choice(fraud_phrases)
        df.at[idx, 'skills_count'] = row['skills_count'] + random.randint(15, 25)
        df.at[idx, 'years_experience'] = random.randint(0, 1)
        processed_texts.append(modified_text)
        labels.append(1) # Flag as Fraud
        
    elif dice_roll < 0.25:
        # 2. CLEVER FRAUD (Hard to catch - Text looks completely normal, numbers are low)
        df.at[idx, 'skills_count'] = random.randint(5, 8) 
        df.at[idx, 'years_experience'] = random.randint(1, 3) 
        processed_texts.append(row['resume_text']) 
        labels.append(1) # Flag as Fraud
        
    elif dice_roll < 0.30:
        # 3. MESSY GENUINE (Over-enthusiastic real applicant who keyword stuffs)
        df.at[idx, 'skills_count'] = row['skills_count'] + random.randint(12, 18)
        processed_texts.append(row['resume_text'])
        labels.append(0) # Flag as GENUINE!
        
    else:
        # 4. STANDARD CLEAN GENUINE
        processed_texts.append(row['resume_text'])
        labels.append(0) # Flag as GENUINE

# Explicitly append the processed list data columns back to the dataframe
df['resume_text'] = processed_texts
df['label'] = labels

# 5. Export clean data matrix now that 'label' is safely inside the index map
final_columns = ['resume_text', 'years_experience', 'skills_count', 'label']
df[final_columns].to_csv("processed_resume_data.csv", index=False)
print("Saved clean matrix schema to 'processed_resume_data.csv'.")

print("\n================== DATA INTEGRITY AUDIT ==================")
audit_summary = df.groupby('label')[['skills_count', 'years_experience']].mean()
distribution = df['label'].value_counts(normalize=True) * 100

print(f"Total Rows Generated: {len(df)}")
print(f"Genuine Profiles (Label 0): {distribution[0]:.2f}% of dataset")
print(f"Suspicious Profiles (Label 1): {distribution[1]:.2f}% of dataset")
print("\nMean Statistical Breakdowns per Group:")
print(audit_summary.to_string())
print("==========================================================")

============ STARTING DATA PREPARATION ============
Reading raw 'Resume.csv' dataset...
Successfully loaded 2484 records from raw data.
Extracting baseline skill counts from unstructured text...
Injecting complex anomalies with statistical noise...
Saved clean matrix schema to 'processed_resume_data.csv'.

================== DATA INTEGRITY AUDIT ==================
Total Rows Generated: 2484
Genuine Profiles (Label 0): 75.93% of dataset
Suspicious Profiles (Label 1): 24.07% of dataset

Mean Statistical Breakdowns per Group:
       skills_count  years_experience
label                                
0          1.708908         10.473489
1         18.322742          0.707358
